# Saz sentezi, birinci aşama v1 (çalış eğrisi → mel) · eğitim · tek hücre

**Yapacağın üç şey:**
1. Üst menü: **Çalışma zamanı → Çalışma zamanı türünü değiştir → A100 GPU → Kaydet**.
2. Aşağıdaki hücrenin solundaki **▶** düğmesine bas.
3. Açılan pencerede Google hesabını seçip **Drive'a izin ver**. **veri:** satırlarını görünce kalkabilirsin.

Gerisi kendiliğinden: veri, eğitim, ölçüm. Eğitim A100'de tahminen 1,5-3 saat; oturum koparsa ya da kota biterse
`DEVAM = 'son'` yapıp yeniden ▶: eğitim son kayıt noktasından (en çok 10 dk geriden) sürer.

- Sonuç Drive'da: `ses_nota/ses-modeli/runs/<tarih>-<AD>/` (en_iyi.pt, dogrulama.jsonl, ornek/, olcu/).
- Ölçüm, Drive'da vokoder-v1 koşusu varsa sesi onunla üretir; yoksa yalnız mel ölçülür.

**İÇ TEST.** Veri ve model yalnız Drive'da kalır; paylaşılmaz.


In [ ]:
# Saz sentezi birinci aşama v1 -- TEK HÜCRE. A100.
# Çalış eğrisi (f0, ses düzeyi, nota olayları, saz) -> mel (44,1 kHz, 128 bant). Akış eşleme: az adımda örneklenir.
# Veri: ses_nota/ses-modeli/vokoder-v1-paket/ + mel-v1-paket/ · kod: ses_nota/ses-modeli/mel-kod.zip
# Çıktı: ses_nota/ses-modeli/runs/<tarih>-<AD>/
AD = 'mel-v1'          # koşu adı: runs/<tarih>-<AD>/ (aynı gün ikinci koşu -2, -3 ...; eski koşunun üstüne yazılmaz)
ADIM = 200000        # eğitim adımı (parti 32 kesit x 3,7 sn)
DEVAM = None          # oturum kesilirse: 'son' (bu AD'li en son koşunun son kayıt noktasından sürer)
KAPAT = True          # bitince Drive'ı kaydedip çalışma zamanını kapat (kota harcamasın)
A100_SART = True      # A100 yoksa eğitime başlamadan dur · False: yine de koş
AYAR = 'orta'        # ağ boyu: orta 5,4 M · hafif 2,4 M
PARTI = 32
BF16 = False          # True: bf16 otomatik karma (daha hızlı; ilk koşuda denenmedi)
# ----------------------------------------------------------------------------------------------------
import torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU yok'
print('GPU:', GPU)
if A100_SART and 'A100' not in GPU:
    raise SystemExit(f'A100 değil ({GPU}): Çalışma zamanı → Çalışma zamanı türünü değiştir → A100 GPU seç, yeniden '
                     'çalıştır. A100 olmadan koşmak istersen A100_SART = False yap.')
from google.colab import drive
drive.mount('/content/drive')
import os, sys, shutil, zipfile
DRIVE = '/content/drive/MyDrive/ses_nota'
SM = f'{DRIVE}/ses-modeli'
KOD = '/content/mel-kod'
shutil.rmtree(KOD, ignore_errors=True)
zipfile.ZipFile(f'{SM}/mel-kod.zip').extractall(KOD)
sys.path.insert(0, KOD)
import mel_colab as mc
_kapat = KAPAT
A = None
try:
    A = mc.ayarlar(AD, ADIM, DEVAM, SM, GPU, KOD, AYAR=AYAR, PARTI=PARTI, BF16=BF16,
                   VOKODER_PAKET='vokoder-v1-paket', MEL_PAKET='mel-v1-paket')
    mc.veri(A)            # iki paket Drive'dan /content'e (boyut denetimiyle) ve açılması
    mc.hazirlik(A)        # koşu klasörü; DEVAM ise son kayıt noktası
    mc.egit(A)            # ADIM'a kadar; 10 dk'da bir son.pt, doğrulamada en iyi olunca en_iyi.pt (Drive'a)
    mc.olc(A)             # ölçüt pasajları gerçek eğrilerden: mel ölçüleri, dikiş; vokoder varsa ses ve perde
except KeyboardInterrupt:  # elle durduruldu: çalışma zamanı açık kalsın
    _kapat = False
    raise
finally:
    if A is not None:
        mc.ozet(A)
    print('BİTTİ.')
    if _kapat:
        print('Drive kaydediliyor, çalışma zamanı kapatılıyor (KAPAT = True)...')
        drive.flush_and_unmount()
        from google.colab import runtime
        runtime.unassign()
    else:
        print('Çalışma zamanı → Çalışma zamanının bağlantısını kes ve sil (A100 boşta da kota harcar).')
